In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from Pruning import *
from scipy.linalg import solve, qr
import numpy as np
from Plot import *
import random
from model import *

# Transform to tensor
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.02
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

train_data = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

test_data = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform
)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

Files already downloaded and verified


/Users/yuekai/miniconda3/envs/CSSP_NN_Pruning/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Files already downloaded and verified


<img src="nn4CIFAR10.png" alt="nn">

In [22]:
class MyCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.model = nn.Sequential(
            # Block 1
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 2
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.Conv2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 4
            nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(512),
            nn.ReLU(),

            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(512),
            nn.ReLU(),

            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(512),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 5
            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(512),
            nn.ReLU(),

            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(512),
            nn.ReLU(),

            nn.Conv2d(in_channels=512, out_channels=512, kernel_size=3, stride=1, padding=1, bias=True),
            nn.BatchNorm2d(512),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2, stride=2),

            # Classifier
            nn.Flatten(),          # (batch_size, 512)
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.model(x)

In [15]:
class Tudui(nn.Module):
    def __init__(self):
        super(Tudui, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, 5, 1, 2),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 32, 5, 1, 2),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 5, 1, 2),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64),
            nn.BatchNorm1d(64),
            #nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.model(x)
        return x

In [23]:
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("mps" if torch.mps.is_available() else "cpu")
#device = torch.device("cpu")

class Model3(nn.Module):
    def __init__(self):

        super(Model3, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool2 = nn.MaxPool2d(2)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool3 = nn.MaxPool2d(2)

        self.fc1 = nn.Linear(64 * 4 * 4, 64)
        self.bn_fc1 = nn.BatchNorm1d(64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        # First layer
        out = self.conv1(x)          # (batch_size, 32, 32, 32)
        out = self.bn1(out)
        out = F.relu(out)
        out = self.pool1(out)        # (batch_size, 32, 16, 16)

        # Save the skip connection branch
        residual = out

        # Main branch of the second layer
        out = self.conv2(out)        # (batch_size, 32, 16, 16)
        out = self.bn2(out)

        # Add before the next ReLU
        out = out + residual
        out = F.relu(out)

        out = self.pool2(out)        # (batch_size, 32, 8, 8)

        # Third layer
        out = self.conv3(out)        # (batch_size, 64, 8, 8)
        out = self.bn3(out)
        out = F.relu(out)
        out = self.pool3(out)        # (batch_size, 64, 4, 4)

        # Linear layers
        out = torch.flatten(out, 1)
        out = self.fc1(out)
        out = self.bn_fc1(out)
        out = F.relu(out)
        out = self.fc2(out)

        return out

model3 = MyCNN().to(device)

# loss functiom & optimizer
loss_fun = nn.CrossEntropyLoss()
optimizer = optim.SGD(model3.parameters(), lr=0.01)

epoch = 20

for i in range(epoch):
    print("-------Epoch {}-------".format(i+1))
    # train
    for imgs, targets in train_loader:
        imgs = imgs.to(device)
        targets = targets.to(device)
        outputs = model3(imgs)
        loss = loss_fun(outputs, targets)

        # optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # test
    total_test_loss = 0
    total_accuracy = 0
    wrong_indices = []
    base_idx = 0
    with torch.no_grad():
        for imgs, targets in test_loader:
            batch_size = targets.size(0)
            imgs = imgs.to(device)
            targets = targets.to(device)
            outputs = model3(imgs)

            loss0 = loss_fun(outputs, targets)
            total_test_loss = total_test_loss + loss0.item()

            accuracy = (outputs.argmax(1) == targets).sum().item()
            total_accuracy = total_accuracy + accuracy

            # Record indices of misclassified samples
            if i == epoch - 1:  # Only record for the last epoch
                wrong_in_batch = torch.where(outputs.argmax(1) != targets)[0]
                wrong_indices.extend((base_idx + wrong_in_batch).tolist())
                base_idx += batch_size


    print("Total loss in test set: {}".format(total_test_loss))
    print("Total accuracy in test set: {}".format(total_accuracy / len(test_data)))

print("Number of samples misclassified by the original model:", len(wrong_indices))

-------Epoch 1-------
Total loss in test set: 201.47828024625778
Total accuracy in test set: 0.5521
-------Epoch 2-------
Total loss in test set: 130.46443313360214
Total accuracy in test set: 0.7105
-------Epoch 3-------
Total loss in test set: 125.50052535533905
Total accuracy in test set: 0.7231
-------Epoch 4-------
Total loss in test set: 108.81344157457352
Total accuracy in test set: 0.7595
-------Epoch 5-------
Total loss in test set: 112.54736050963402
Total accuracy in test set: 0.7492
-------Epoch 6-------
Total loss in test set: 97.9599494934082
Total accuracy in test set: 0.7909
-------Epoch 7-------
Total loss in test set: 91.0627036690712
Total accuracy in test set: 0.8017
-------Epoch 8-------
Total loss in test set: 87.64051577448845
Total accuracy in test set: 0.8088
-------Epoch 9-------
Total loss in test set: 87.24207007884979
Total accuracy in test set: 0.8132
-------Epoch 10-------
Total loss in test set: 92.050965487957
Total accuracy in test set: 0.8018
-------E

In [24]:
for i in range(epoch):
    print("-------Epoch {}-------".format(i+21))
    # train
    for imgs, targets in train_loader:
        imgs = imgs.to(device)
        targets = targets.to(device)
        outputs = model3(imgs)
        loss = loss_fun(outputs, targets)

        # optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # test
    total_test_loss = 0
    total_accuracy = 0
    wrong_indices = []
    base_idx = 0
    with torch.no_grad():
        for imgs, targets in test_loader:
            batch_size = targets.size(0)
            imgs = imgs.to(device)
            targets = targets.to(device)
            outputs = model3(imgs)

            loss0 = loss_fun(outputs, targets)
            total_test_loss = total_test_loss + loss0.item()

            accuracy = (outputs.argmax(1) == targets).sum().item()
            total_accuracy = total_accuracy + accuracy


    print("Total loss in test set: {}".format(total_test_loss))
    print("Total accuracy in test set: {}".format(total_accuracy / len(test_data)))

-------Epoch 21-------
Total loss in test set: 77.81142862141132
Total accuracy in test set: 0.8383
-------Epoch 22-------
Total loss in test set: 68.27734090387821
Total accuracy in test set: 0.8611
-------Epoch 23-------
Total loss in test set: 67.09408695995808
Total accuracy in test set: 0.8596
-------Epoch 24-------
Total loss in test set: 65.91048489511013
Total accuracy in test set: 0.8679
-------Epoch 25-------
Total loss in test set: 65.09678848087788
Total accuracy in test set: 0.8683
-------Epoch 26-------
Total loss in test set: 72.03102964162827
Total accuracy in test set: 0.8593
-------Epoch 27-------
Total loss in test set: 68.72630209475756
Total accuracy in test set: 0.8648
-------Epoch 28-------
Total loss in test set: 70.3320010304451
Total accuracy in test set: 0.869
-------Epoch 29-------
Total loss in test set: 63.17524107545614
Total accuracy in test set: 0.8725
-------Epoch 30-------
Total loss in test set: 70.67831341549754
Total accuracy in test set: 0.8654
--

In [25]:
for i in range(epoch):
    print("-------Epoch {}-------".format(i+41))
    # train
    for imgs, targets in train_loader:
        imgs = imgs.to(device)
        targets = targets.to(device)
        outputs = model3(imgs)
        loss = loss_fun(outputs, targets)

        # optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # test
    total_test_loss = 0
    total_accuracy = 0
    wrong_indices = []
    base_idx = 0
    with torch.no_grad():
        for imgs, targets in test_loader:
            batch_size = targets.size(0)
            imgs = imgs.to(device)
            targets = targets.to(device)
            outputs = model3(imgs)

            loss0 = loss_fun(outputs, targets)
            total_test_loss = total_test_loss + loss0.item()

            accuracy = (outputs.argmax(1) == targets).sum().item()
            total_accuracy = total_accuracy + accuracy


    print("Total loss in test set: {}".format(total_test_loss))
    print("Total accuracy in test set: {}".format(total_accuracy / len(test_data)))

-------Epoch 41-------
Total loss in test set: 77.58133049309254
Total accuracy in test set: 0.864
-------Epoch 42-------
Total loss in test set: 68.93775932490826
Total accuracy in test set: 0.8779
-------Epoch 43-------
Total loss in test set: 63.896452225744724
Total accuracy in test set: 0.8857
-------Epoch 44-------
Total loss in test set: 67.99749106168747
Total accuracy in test set: 0.8811
-------Epoch 45-------
Total loss in test set: 67.89568804949522
Total accuracy in test set: 0.8817
-------Epoch 46-------
Total loss in test set: 66.85430792719126
Total accuracy in test set: 0.883
-------Epoch 47-------
Total loss in test set: 75.98963618278503
Total accuracy in test set: 0.8686
-------Epoch 48-------
Total loss in test set: 68.72208796441555
Total accuracy in test set: 0.88
-------Epoch 49-------
Total loss in test set: 71.26351162791252
Total accuracy in test set: 0.8781
-------Epoch 50-------
Total loss in test set: 74.26960483938456
Total accuracy in test set: 0.8757
---

In [49]:
import importlib
import Pruning
importlib.reload(Pruning)
from Pruning import *
batch_list = list(train_loader)
selected_batches = random.sample(batch_list, 8)  
X = torch.cat([batch[0] for batch in selected_batches], dim=0)
X = X.to(device)
pruned_model_iter = iterative_pruning(model3, X, (3,32,32), 0.99999999, 0.98, "ARP")

acc, wrong_samples_pruned = evaluate_pruned_model(pruned_model_iter, test_loader)

print()
print(acc, len(wrong_samples_pruned))

-------Begin pruning-------
layer_idx: 40, layer_type: Conv2d
number of out_channels: 512 -> 501

Flops after pruning: 627586816 -> 627169960

0.1 9000
